# Assessing EUMETSAT Metop-SG Products for Cloud-native Access


**Authors**: Rajat Shinde (UAH), Harshini Girish (UAH), Alex Mandel (Development Seed), Brian Freitag (NASA MSFC)

**Date**: September 16, 2026

**Description**: Metop-SGA1 carries six instrument missions: METimage (VII),
IASI-NG, MWS, a Radio Occultation sounder, 3MI, and the Copernicus
Sentinel-5/UVNS spectrometer. Their products are distributed through the
EUMETSAT Data Store as zip packages containing a data file and two XML
sidecars. This notebook checks, for one recent granule per instrument, how
close each product is to cloud-native access.

**Setup**: This notebook will:

1. Find the Data Store collection for each instrument
2. Check whether the hosting answers HTTP range requests
3. Open a granule lazily over HTTP with xarray, without downloading it
4. Download one granule and read its chunking and compression settings
5. Build a reference file with [VirtualiZarr](https://virtualizarr.readthedocs.io/) (and optionally write kerchunk JSON) so the remote granule can be read through the Zarr engine


## Run this notebook

You need a free EUMETSAT account. Register at
[user.eumetsat.int](https://user.eumetsat.int), then copy your consumer key
and secret from [api.eumetsat.int/api-key](https://api.eumetsat.int/api-key).
The token they produce expires after about an hour; re-run the credentials
cell if requests start returning 401.

In [ ]:
%pip install -q eumdac "xarray>=2024.10" h5netcdf fsspec aiohttp kerchunk zarr pandas requests zstandard virtualizarr obstore


Note: you may need to restart the kernel to use updated packages.


In [16]:
import json, shutil, urllib.parse, zlib
from pathlib import Path
import numpy as np, pandas as pd, requests, fsspec, xarray as xr, eumdac

KEY, SECRET = Path.home().joinpath(".eumdac", "credentials").read_text().strip().split(",")
token = eumdac.AccessToken((KEY, SECRET))
store = eumdac.DataStore(token)

def auth():
    return {"Authorization": f"Bearer {token}"}   # str(token) auto-refreshes

DATA = Path("data"); DATA.mkdir(exist_ok=True)
print("token ok")

token ok


## About the datasets

The six products assessed here are the ones listed on the
[Metop-SG test data page](https://user.eumetsat.int/resources/user-guides/metop-sg-test-data):
one per instrument on Metop-SGA1.

| Instrument | Measures | Product |
|---|---|---|
| METimage (VII) | visible and infrared radiances, 20 channels | level 1B radiances |
| MWS | microwave sounding, 24 channels | level 1B |
| Radio Occultation (GRAS-2) | GNSS bending angles | level 1B |

They come through two different doors, and the notebook handles both:

- **Data Store collections** exist for the instruments already distributing
  flight data. As of this writing that is METimage (`EO:EUM:DAT:0464`),
  MWS (`EO:EUM:DAT:0450`) and GRAS-2 radio occultation (`EO:EUM:DAT:0452`).
- **Test-data downloads** are direct links on the page above, for the
  instruments not yet in the Data Store (IASI-NG, 3MI, Sentinel-5) and as
  pre-launch samples for the others. Open the page in a browser, copy each
  product's download link, and paste it below.

The distinction is itself part of the assessment: a test-data link tells you
about the file format EUMETSAT intends to ship, while only a Data Store
collection tells you about the hosting the operational data will live behind.


In [17]:
# Each product is either a Data Store collection ID or a direct download
# link copied from the Metop-SG test data page. Fill in the missing links.
PRODUCTS = {
    "METimage (VII)":    {"collection": "EO:EUM:DAT:0464"},
    "MWS":               {"collection": "EO:EUM:DAT:0450"},
    "Radio Occultation": {"collection": "EO:EUM:DAT:0452"},
}

## Helper functions

Short helpers used in every section. Prefer the function docstrings below over
re-describing them here.


In [18]:
enc = lambda s: urllib.parse.quote(s, safe="")

def latest(collection_id):
    """Newest product in a collection, plus the files in its package."""
    prod = store.get_collection(collection_id).search().first()
    entries = list(prod.entries)
    print(prod, "\n  files:", entries)
    return prod, entries

def entry_url(prod, filename):
    """Direct URL for one file inside the product, bypassing the zip."""
    return f"{prod.url.split('?')[0]}/entry?name={enc(filename)}"

def ranges_ok(url):
    """True when the server answers 206 for both a leading and a suffix range."""
    codes = {}
    for label, rng in (("head", "bytes=0-1023"), ("tail", "bytes=-65536")):
        r = requests.get(url, headers={**auth(), "Range": rng}, stream=True, timeout=60)
        codes[label] = r.status_code
        if label == "head":
            print("first bytes:", r.raw.read(8).hex(), " (894844... means HDF5/netCDF-4)")
        r.close()
    print("range status:", codes)
    return codes["head"] == 206 and codes["tail"] == 206

def fetch(url, filename=None):
    """Download a direct test-data link. Zips need unzipping afterwards."""
    dest = DATA / (filename or url.split("/")[-1].split("?")[0])
    if not dest.exists():
        with requests.get(url, headers=auth(), stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                shutil.copyfileobj(r.raw, f)
    print(f"{dest.name}: {dest.stat().st_size/1e6:.0f} MB")
    return dest

def remote_size(url):
    """File size via a 1-byte ranged GET, since the endpoint may not do HEAD."""
    r = requests.get(url, headers={**auth(), "Range": "bytes=0-0"},
                     stream=True, timeout=60)
    r.close()
    if r.status_code == 206:
        return int(r.headers["Content-Range"].split("/")[-1])
    return int(r.headers["Content-Length"])

def open_remote(url, log_requests=True):
    """Lazy-open a remote netCDF-4 over HTTP. Downloads nothing.

    Uses fsspec blockcache so ``block_size`` actually batches range reads.
    When ``log_requests`` is True, prints how many range requests and how many
    bytes the open itself required.
    """
    fs = fsspec.filesystem("https", client_kwargs={"headers": auth()},
                           encoded=True, skip_instance_cache=True)
    f = fs.open(url, mode="rb", block_size=4 * 2**20, cache_type="blockcache",
                size=remote_size(url))
    tree = xr.open_datatree(f, engine="h5netcdf", phony_dims="access",
                            decode_times=False)
    if log_requests and getattr(f, "cache", None) is not None:
        print(f"open_remote: {f.cache.miss_count} range requests, "
              f"{f.cache.total_requested_bytes / 1e6:.1f} MB transferred")
    return tree

def download(prod, filename):
    """Fetch one file from the product through the entry endpoint."""
    dest = DATA / filename
    if not dest.exists():
        with prod.open(entry=filename) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
    print(f"{dest.name}: {dest.stat().st_size/1e6:.0f} MB")
    return dest

def layout(path):
    """Chunk shape, chunk size, and codec for every variable in the file."""
    tree = xr.open_datatree(path, engine="h5netcdf", phony_dims="access", decode_times=False)
    rows = []
    for node in tree.subtree:
        for name, v in node.ds.data_vars.items():
            ch = v.encoding.get("chunksizes")
            rows.append({"variable": f"{node.path}/{name}", "shape": tuple(v.shape),
                         "MB": round(v.nbytes / 1e6, 2),
                         "chunks": tuple(ch) if ch else None,
                         "chunk_MB": round(v.dtype.itemsize * int(np.prod(ch)) / 1e6, 3) if ch else None,
                         "codec": v.encoding.get("compression"),
                         "shuffle": bool(v.encoding.get("shuffle"))})
    return pd.DataFrame(rows).sort_values("MB", ascending=False)


## METimage (VII) level 1B radiances

The full walkthrough. Later instruments repeat the discovery → range check →
lazy open → layout steps. METimage also covers read-amplification timing and
VirtualiZarr/kerchunk references.

First, the newest granule and its package contents. Expect one `.nc` and two
XML sidecars: the zip exists to carry those sidecars.


In [19]:
spec = PRODUCTS["METimage (VII)"]
prod, entries = latest(spec["collection"])
nc = next(e for e in entries if e.endswith(".nc"))
url = entry_url(prod, nc)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EUMT_20260922162541_G_O_20260922160859_20260922161000_C_N_T__ 
  files: ['W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EUMT_20260922162541_G_O_20260922160859_20260922161000_C_N_T__.nc', 'EOPMetadata.xml', 'manifest.xml']


Check range support on the per-file URL. Without 206 responses here, nothing
else in this notebook is possible and the product is download-only.

In [20]:
ranges_ok(url)

first bytes: 894844460d0a1a0a  (894844... means HDF5/netCDF-4)
range status: {'head': 206, 'tail': 206}


True

Open the granule over HTTP. This reads metadata by range request and defers
everything else, so it should finish in seconds even though the file is over
100 MB. The request log from ``open_remote`` shows how chatty that metadata
pass is.


In [21]:
%time tree = open_remote(url)
tree

open_remote: 2 range requests, 8.4 MB transferred
CPU times: user 742 ms, sys: 18.8 ms, total: 761 ms
Wall time: 20.1 s


<xarray.DataTree>
Group: /
│   Attributes: (12/21)
│       title:                   VII L1B Radiances
│       Conventions:             CF-1.6
│       metadata_conventions:    Unidata Dataset Discovery v1.0
│       product_name:            W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EU...
│       summary:                 VII/METimage L1B top of the atmosphere radiances
│       doi:                     
│       ...                      ...
│       sensing_start_time_utc:  2026-09-22 16:08:59.725
│       sensing_end_time_utc:    2026-09-22 16:10:00.215
│       environment:             Operational
│       references:              www.eumetsat.int
│       orbit_start:             5764
│       orbit_end:               5764
├── Group: /status
│   ├── Group: /status/satellite
│   │       Dimensions:                   ()
│   │       Data variables: (12/24)
│   │           epoch_time_utc            float64 8B ...
│   │           semi_major_axis           float64 8B ...
│   │           eccentricity              float64 8B ...
│   │           inclination               float64 8B ...
│   │           perigee_argument          float64 8B ...
│   │           right_ascension           float64 8B ...
│   │           ...                        ...
│   │           z_velocity                float64 8B ...
│   │           yaw_error                 float64 8B ...
│   │           roll_error                float64 8B ...
│   │           pitch_error               float64 8B ...
│   │           leap_second_time_utc      float64 8B ...
│   │           leap_second_value         float32 4B ...
│   ├── Group: /status/instrument
│   │       Dimensions:              (mode_items: 1)
│   │       Dimensions without coordinates: mode_items
│   │       Data variables:
│   │           mode_start_time_utc  (mode_items) float64 8B ...
│   │           mode_end_time_utc    (mode_items) float64 8B ...
│   │           instrument_mode      (mode_items) <U4 16B ...
│   └── Group: /status/processing
│           Dimensions:            ()
│           Data variables:
│               creation_time_utc  float64 8B ...
│           Attributes:
│               processor_name:              VII_L1B
│               processor_version:           1.0
│               processing_mode:             NRT
│               format_version:              6.0
│               auxiliary_data_version:      EUM/LEO-EPSSG/SPE/14/777147 v4A
│               pgs_reference_and_version:   EUM/LEO-EPSSG/DOC/14/746628 v5
│               pfs_reference_and_version:   EUM/LEO-EPSSG/SPE/14/777138 v5
│               atbd_reference_and_version:  EUM/LEO-EPSSG/DOC/13/702485 v4
│               source:                      ['SGA1_VII_1B_AUX_LMDB___S20250813000000Z_Ex...
├── Group: /data
│   ├── Group: /data/measurement_data
│   │       Dimensions:              (num_tie_points_alt: 140, num_tie_points_act: 394,
│   │                                 num_lines: 840, num_pixels: 3144, num_scans: 35)
│   │       Dimensions without coordinates: num_tie_points_alt, num_tie_points_act,
│   │                                       num_lines, num_pixels, num_scans
│   │       Data variables: (12/32)
│   │           latitude             (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           longitude            (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           delta_lat_N_dem      (num_lines, num_pixels) float32 11MB ...
│   │           delta_lon_E_dem      (num_lines, num_pixels) float32 11MB ...
│   │           solar_zenith         (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           solar_azimuth        (num_tie_points_alt, num_tie_points_act) float64 441kB ...
│   │           ...                   ...
│   │           vii_6725             (num_lines, num_pixels) float32 11MB ...
│   │           vii_7325             (num_lines, num_pixels) float32 11MB ...
│   │           vii_8540             (num_lines, num_pixels) float32 11MB ...
│   │           vii_10690     

Pull a 64×64 slice of one radiance channel to confirm that subsetting works
and to feel the per-read latency.


In [22]:
rad = tree["data/measurement_data"].ds
channel = [v for v in rad.data_vars if "radiance" in v.lower() or "vii_" in v.lower()][0]
%time sample = rad[channel][:64, :64].values
print(channel, sample.shape, sample.dtype)

CPU times: user 45.4 ms, sys: 3.47 ms, total: 48.8 ms
Wall time: 10.5 s
vii_443 (64, 64) float32


Download the file once and read its storage layout. Three things to look for
in the table:

1. **Chunking** — whether large variables are chunked at all
2. **Chunk size** — whether sizes land near useful targets (about 1–4 MB for
   interactive reads, 1–16 MB for agentic subsetting, 32–64 MB for training)
3. **Codec** — whether any compression filter is set inside the netCDF

A `codec` column full of `None` means there is **no compression within the
netCDF** (the on-disk file may still be smaller than the sum of `nbytes` for
other reasons, such as sparse fill or unallocated extents, but HDF5 filters
are not compressing the arrays).


In [23]:
local = download(prod, nc)
L = layout(local)
L.head(15)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-VII-1B-RAD_C_EUMT_20260922162541_G_O_20260922160859_20260922161000_C_N_T__.nc: 119 MB


,variable,shape,MB,chunks,chunk_MB,codec,shuffle
46,/data/measurement_data/vii_668,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
45,/data/measurement_data/vii_555,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
34,/data/measurement_data/delta_lat_N_dem,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
35,/data/measurement_data/delta_lon_E_dem,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
50,/data/measurement_data/vii_914,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
48,/data/measurement_data/vii_763,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
49,/data/measurement_data/vii_865,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
47,/data/measurement_data/vii_752,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
44,/data/measurement_data/vii_443,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False
60,/data/measurement_data/vii_8540,"(840, 3144)",10.56,"(1, 3144)",0.013,None,False


Summarize the layout table. `nbytes` / in-memory size is the sum of each
array's `dtype.itemsize * nelements` as Python would hold it uncompressed.
That is not always identical to an HDF5 "logical size," but it is the right
baseline to compare against the on-disk file size. `on disk` is the actual
file size. Tiny median `chunk_MB` values mean remote subsetting will issue
many small range requests.


In [24]:
print("compressed variables:", int(L.codec.notna().sum()), "of", len(L))
print("in-memory size (sum of nbytes):", round(L.MB.sum()), "MB   on disk:",
      round(local.stat().st_size / 1e6), "MB")
print("chunk sizes (MB):", L.chunk_MB.describe()[["min", "50%", "max"]].round(3).to_dict())


compressed variables: 0 of 115
in-memory size (sum of nbytes): 235 MB   on disk: 119 MB
chunk sizes (MB): {'min': 0.0, '50%': 0.013, 'max': 0.013}


Measure what internal compression would buy. This compresses **one on-disk
chunk** on local CPU, with and without an HDF5-style byte shuffle (bytes of
each element grouped by significance).

On many float fields, deflate level 1 with shuffle is the cheap, widely
readable option and zstd does the same job faster. Ratios above about 1.5×
make a solid case. If shuffle *hurts* on a given chunk, that often means the
native byte layout already has long runs (fills, constants) that shuffle
scatters — report the better of the two rather than assuming shuffle always wins.


In [25]:
import zstandard as zstd

def hdf5_byte_shuffle(arr):
    """HDF5 shuffle: group byte-0 of every element, then byte-1, ..."""
    a = np.ascontiguousarray(arr)
    return a.view(np.uint8).reshape(-1, a.dtype.itemsize).ravel(order="F").tobytes()

row = L.dropna(subset=["chunks"]).iloc[0]
node = tree[str(Path(row.variable).parent)].ds
name = Path(row.variable).name
# one storage chunk, not an arbitrary 512² window
slices = tuple(slice(0, c) for c in row.chunks)
arr = np.ascontiguousarray(node[name][slices].values)
raw = arr.tobytes()
shuf = hdf5_byte_shuffle(arr)

print(row.variable, arr.shape, arr.dtype, "chunk", row.chunks)
print(f"deflate-1: {len(raw)/len(zlib.compress(raw, 1)):.2f}x, "
      f"with shuffle {len(raw)/len(zlib.compress(shuf, 1)):.2f}x")
c = zstd.ZstdCompressor(level=3)
print(f"zstd-3:    {len(raw)/len(c.compress(raw)):.2f}x, "
      f"with shuffle {len(raw)/len(c.compress(shuf)):.2f}x")


/data/measurement_data/vii_668 (1, 3144) float32 chunk (1, 3144)
deflate-1: 141.30x, with shuffle 149.71x
zstd-3:    571.64x, with shuffle 465.78x


### Read efficiency and virtual references

Still on the METimage granule from above: compare a local slice to the same
slice over HTTP, counting range requests, then build a VirtualiZarr reference
file (kerchunk JSON) so subsequent opens skip the expensive HDF5 metadata walk.

These cells reuse `url`, `local`, `auth`, and `remote_size` from the helper
section rather than hard-coding a filename.


In [26]:
import time

# Target mid-swath so we are not reading a corner that happens to be fill.
rad = tree["data/measurement_data"].ds
var = next(v for v in rad.data_vars if v.startswith("vii_"))
ny, nx = rad[var].shape[:2]
sl = (slice(ny // 2, ny // 2 + 64), slice(nx // 2, nx // 2 + 64))
chunks = rad[var].encoding.get("chunksizes")
print(f"variable : {var} {rad[var].shape} {rad[var].dtype}")
print(f"chunks   : {chunks}  codec={rad[var].encoding.get('compression')}")
print(f"file     : {local.stat().st_size / 1e6:.0f} MB\n")

# ---- local baseline -------------------------------------------------------
t0 = time.perf_counter()
a = xr.open_datatree(local, engine="h5netcdf", phony_dims="access",
                     decode_times=False)["data/measurement_data"].ds[var][sl].values
t_local = time.perf_counter() - t0
wanted = a.nbytes
print(f"local slice : {wanted / 1e3:.0f} KB in {t_local * 1e3:.0f} ms\n")

# ---- remote: reopen with blockcache so miss_count is meaningful ----------
bs = 4 * 2**20
fs = fsspec.filesystem("https", client_kwargs={"headers": auth()},
                       encoded=True, skip_instance_cache=True)
f = fs.open(url, mode="rb", block_size=bs, cache_type="blockcache",
            size=remote_size(url))

t0 = time.perf_counter()
rt = xr.open_datatree(f, engine="h5netcdf", phony_dims="access", decode_times=False)
t_open = time.perf_counter() - t0
open_bytes, open_reqs = f.cache.total_requested_bytes, f.cache.miss_count

t0 = time.perf_counter()
b = rt["data/measurement_data"].ds[var][sl].values
t_read = time.perf_counter() - t0
read_bytes = f.cache.total_requested_bytes - open_bytes
read_reqs = f.cache.miss_count - open_reqs

print(f"open  : {open_bytes / 1e6:6.1f} MB in {open_reqs:3d} reads, {t_open:5.1f}s")
print(f"slice : {read_bytes / 1e6:6.1f} MB in {read_reqs:3d} reads, {t_read:5.1f}s"
      f"   -> {read_bytes / wanted:.0f}x amplification for {wanted / 1e3:.0f} KB wanted")
print(f"total : {(open_bytes + read_bytes) / 1e6:6.1f} MB of a "
      f"{local.stat().st_size / 1e6:.0f} MB file")
assert np.allclose(a, b, equal_nan=True)


variable : vii_443 (840, 3144) float32
chunks   : (1, 3144)  codec=None
file     : 119 MB

local slice : 16 KB in 710 ms

open  :    8.4 MB in   2 reads,   7.2s
slice :    4.2 MB in   1 reads,   1.8s   -> 256x amplification for 16 KB wanted
total :   12.6 MB of a 119 MB file


### VirtualiZarr → kerchunk references

VirtualiZarr reads the local netCDF once and writes a small reference file.
Readers then use that sidecar (kerchunk JSON) to fetch only the bytes they
need from the remote URL — without re-walking HDF5 metadata on every open.



In [27]:
# Index the local download, then rename chunk paths to the remote Data Store URL.
sample = DATA / "metimage_l1b_sample.nc"
if not sample.exists():
    shutil.copy2(local, sample)
print(sample, f"{sample.stat().st_size / 1e6:.0f} MB")
print("remote url:", url)


data/metimage_l1b_sample.nc 119 MB
remote url: https://api.eumetsat.int/data/download/1.0.0/collections/EO%3AEUM%3ADAT%3A0464/products/W_XX-EUMETSAT-Darmstadt%2CSAT%2CSGA1-VII-1B-RAD_C_EUMT_20260922162541_G_O_20260922160859_20260922161000_C_N_T__/entry?name=W_XX-EUMETSAT-Darmstadt%2CSAT%2CSGA1-VII-1B-RAD_C_EUMT_20260922162541_G_O_20260922160859_20260922161000_C_N_T__.nc


In [28]:
import obstore
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry

registry = ObjectStoreRegistry({DATA.resolve().as_uri(): obstore.store.LocalStore(str(DATA.resolve()))})

vds = open_virtual_dataset(
    sample.resolve().as_uri(),
    parser=HDFParser(group="data/measurement_data"),
    registry=registry,
    decode_times=False,
)
vds


<xarray.Dataset> Size: 118MB
Dimensions:              (num_tie_points_alt: 140, num_tie_points_act: 394,
                          data/num_lines: 840, data/num_pixels: 3144,
                          data/num_scans: 35)
Dimensions without coordinates: num_tie_points_alt, num_tie_points_act,
                                data/num_lines, data/num_pixels, data/num_scans
Data variables: (12/32)
    latitude             (num_tie_points_alt, num_tie_points_act) int32 221kB ManifestArray<shape=(140, 394), dtype=int32, chunks=(1, 394)...
    longitude            (num_tie_points_alt, num_tie_points_act) uint32 221kB ManifestArray<shape=(140, 394), dtype=uint32, chunks=(1, 394...
    delta_lat_N_dem      (data/num_lines, data/num_pixels) int16 5MB Manifest...
    delta_lon_E_dem      (data/num_lines, data/num_pixels) int16 5MB Manifest...
    solar_zenith         (num_tie_points_alt, num_tie_points_act) uint32 221kB ManifestArray<shape=(140, 394), dtype=uint32, chunks=(1, 394...
    solar_azimuth        (num_tie_points_alt, num_tie_points_act) uint32 221kB ManifestArray<shape=(140, 394), dtype=uint32, chunks=(1, 394...
    ...                   ...
    vii_6725             (data/num_lines, data/num_pixels) uint16 5MB Manifes...
    vii_7325             (data/num_lines, data/num_pixels) uint16 5MB Manifes...
    vii_8540             (data/num_lines, data/num_pixels) uint16 5MB Manifes...
    vii_10690            (data/num_lines, data/num_pixels) uint16 5MB Manifes...
    vii_12020            (data/num_lines, data/num_pixels) uint16 5MB Manifes...
    vii_13345            (data/num_lines, data/num_pixels) uint16 5MB Manifes...

In [29]:
# Point every chunk at the remote URL (byte offsets stay the same).
vds_remote = vds.vz.rename_paths(url)

refs_path = Path("metimage_refs.json")
vds_remote.vz.to_kerchunk(str(refs_path), format="json")
print(f"{refs_path.stat().st_size / 1e6:.2f} MB sidecar for a "
      f"{sample.stat().st_size / 1e6:.0f} MB granule")

# Zarr v3 needs the reference FS and its HTTPS target both asynchronous.
fs = fsspec.filesystem(
    "reference",
    fo=str(refs_path),
    remote_protocol="https",
    asynchronous=True,
    remote_options={"headers": auth(), "encoded": True, "asynchronous": True},
)
ds = xr.open_dataset(
    fs.get_mapper(""),
    engine="zarr",
    consolidated=False,
    decode_times=False,
)
%time sample_remote = ds[var][sl].values
print(var, sample_remote.shape, "allclose",
      np.allclose(a, sample_remote, equal_nan=True))


7.22 MB sidecar for a 119 MB granule
CPU times: user 73.1 ms, sys: 979 μs, total: 74.1 ms
Wall time: 11.6 s
vii_443 (64, 64) allclose True


## MWS

Microwave sounder, 24 channels, small granules. The same steps usually run in
under a minute end to end.

In [30]:
spec = PRODUCTS["MWS"]
if spec.get("collection"):
    prod, entries = latest(spec["collection"])
    datafile = next(e for e in entries if not e.lower().endswith((".xml", ".txt")))
    url = entry_url(prod, datafile)
else:
    url = spec["url"]                     # direct test-data link
    assert url, "paste this product's link from the test-data page into PRODUCTS"

ranges_ok(url)

W_XX-EUMETSAT-Darmstadt,SAT,SGA1-MWS-1B-RAD_C_EUMT_20260922161338_G_O_20260922160559_20260922160857_O_N____ 
  files: ['W_XX-EUMETSAT-Darmstadt,SAT,SGA1-MWS-1B-RAD_C_EUMT_20260922161338_G_O_20260922160559_20260922160857_O_N____.nc', 'EOPMetadata.xml', 'manifest.xml']
first bytes: 894844460d0a1a0a  (894844... means HDF5/netCDF-4)
range status: {'head': 206, 'tail': 206}


True

In [31]:
tree = open_remote(url)
local = download(prod, datafile) if spec.get("collection") else fetch(url)
L = layout(local)
print("compressed:", int(L.codec.notna().sum()), "of", len(L),
      "  median chunk MB:", L.chunk_MB.median())
L.head(10)

open_remote: 1 range requests, 4.2 MB transferred
W_XX-EUMETSAT-Darmstadt,SAT,SGA1-MWS-1B-RAD_C_EUMT_20260922161338_G_O_20260922160559_20260922160857_O_N____.nc: 3 MB
compressed: 0 of 167   median chunk MB: 0.1065


,variable,shape,MB,chunks,chunk_MB,codec,shuffle
130,/data/calibration/mws_toa_radiance,"(79, 95, 24)",1.44,"(28, 95, 24)",0.511,None,False
108,/data/calibration/mws_toa_brightness_temperature,"(79, 95, 24)",1.44,"(28, 95, 24)",0.511,None,False
136,/data/measurement/mws_earth_view_counts,"(79, 95, 24)",0.72,"(57, 95, 24)",0.520,None,False
141,/data/processing_information/mws_radiance_flag,"(79, 95, 24)",0.18,"(79, 95, 24)",0.180,None,False
162,/data/processing_information/mws_brightnesstem...,"(79, 95, 24)",0.18,"(79, 95, 24)",0.180,None,False
99,/data/navigation/mws_surface_type,"(79, 95, 2)",0.12,None,NaN,None,False
96,/data/navigation/mws_solar_azimuth_angle,"(79, 95)",0.06,None,NaN,None,False
95,/data/navigation/mws_satellite_zenith_angle,"(79, 95)",0.06,None,NaN,None,False
135,/data/measurement/mws_earth_view_counts_os_stdev,"(79, 95, 2)",0.06,None,NaN,None,False
94,/data/navigation/mws_solar_zenith_angle,"(79, 95)",0.06,None,NaN,None,False


## Radio Occultation

GRAS-2 has a Data Store collection, and its title says netCDF. The
`ranges_ok` call prints the first bytes as a check anyway: `894844` opens
like the others, `425546` spells BUFR, which has no internal chunking to
assess and would end this section early as its own finding.

If the file is netCDF-4, continue with the same `open_remote` / `download` /
`layout` pattern used for MWS.


In [32]:
spec = PRODUCTS["Radio Occultation"]
if spec.get("collection"):
    prod, entries = latest(spec["collection"])
    datafile = next(e for e in entries if not e.lower().endswith((".xml", ".txt")))
    url = entry_url(prod, datafile)
else:
    url = spec["url"]                     # direct test-data link
    assert url, "paste this product's link from the test-data page into PRODUCTS"

ok = ranges_ok(url)
if not ok:
    print("range requests not supported — download-only for this product")


W_XX-EUMETSAT-Darmstadt,SAT,SGA1-RO_-1B-BND_C_EUMT_20260922164656_G_O_20260922155827_20260922160416_O_N_C20 
  files: ['W_XX-EUMETSAT-Darmstadt,SAT,SGA1-RO_-1B-BND_C_EUMT_20260922164656_G_O_20260922155827_20260922160416_O_N_C20.nc', 'EOPMetadata.xml', 'manifest.xml']
first bytes: 894844460d0a1a0a  (894844... means HDF5/netCDF-4)
range status: {'head': 206, 'tail': 206}
